# Bài học 17: Database — Lưu trữ dữ liệu

## Tại sao cần database?

Ở các bài trước, khi agent tạo xong một bài viết, kết quả chỉ hiện lên màn hình rồi **biến mất**. Tắt chương trình là mất sạch.

Trong thực tế, chúng ta cần:
- **Lưu trữ** các bài viết đã tạo
- **Theo dõi** trạng thái (đang xử lý, hoàn thành, lỗi...)
- **Tìm kiếm** và **lọc** bài viết theo tiêu chí
- **Xem lại** lịch sử chỉnh sửa

**Database** giải quyết tất cả những điều này.

## Tại sao Airtable?

Sản phẩm của chúng ta sử dụng **Airtable** làm database chính. Airtable là dịch vụ database trên cloud với **giao diện bảng tính**:

- **Giao diện bảng tính** — xem, lọc, sắp xếp dữ liệu trực quan như Google Sheets
- **Không cần cài đặt file** — dữ liệu lưu trên cloud, truy cập từ mọi nơi
- **API đầy đủ** — đọc/ghi dữ liệu bằng Python qua thư viện `pyairtable`
- **Chia sẻ dễ dàng** — cả team có thể xem tiến trình bài viết theo thời gian thực

So sánh:
- **SQLite** (database cũ): File cục bộ, chỉ truy cập từ máy chạy code. Giờ chỉ dùng cho lưu lịch sử chat.
- **Airtable** (database chính): Cloud, giao diện web, chia sẻ với team, API dễ dùng.

In [ ]:
# Airtable lưu trữ dữ liệu dưới dạng bảng, giống bảng tính
# Mỗi hàng là một "record" (bản ghi), mỗi cột là một "field" (trường)
# Record ID trong Airtable là CHUỖI (ví dụ: "recABC123"), không phải số nguyên

# Ví dụ minh họa cấu trúc dữ liệu (không cần API key)
sample_articles = [
    {"id": "recABC123", "topic": "SEO Guide 2026", "status": "queued"},
    {"id": "recDEF456", "topic": "Content Marketing", "status": "review"},
    {"id": "recGHI789", "topic": "Link Building", "status": "writing"},
]

print("Cấu trúc dữ liệu bài viết trong Airtable:")
print()
for article in sample_articles:
    print(f"  ID: {article['id']}  |  Topic: {article['topic']}  |  Status: {article['status']}")

print()
print("Lưu ý: ID là chuỗi (recABC123), không phải số nguyên (1, 2, 3)")
print("Đây là khác biệt quan trọng so với SQLite.")

## Airtable cơ bản

Airtable tổ chức dữ liệu theo cấu trúc:
- **Base** — Một dự án (giống một file database)
- **Table** — Một bảng trong base (giống sheet trong bảng tính)
- **Record** — Một hàng trong bảng (giống một bài viết)
- **Field** — Một cột trong bảng (giống topic, status, word_count)

### Thao tác cơ bản với pyairtable

| Thao tác | Hàm pyairtable | Mô tả |
|----------|----------------|-------|
| Tạo record | `table.create(fields)` | Thêm hàng mới |
| Đọc record | `table.get(record_id)` | Lấy một hàng theo ID |
| Liệt kê | `table.all()` | Lấy tất cả hàng |
| Cập nhật | `table.update(record_id, fields)` | Sửa dữ liệu |
| Lọc | `table.all(formula="...")` | Tìm theo điều kiện |

### Record ID
Mỗi record trong Airtable có một **Record ID** dạng chuỗi (ví dụ: `"recABC123"`). Đây là khác biệt lớn so với SQLite, nơi ID là số nguyên tự động tăng (1, 2, 3...).

Trong code sản phẩm, mọi `article_id` đều là **chuỗi**, không phải số nguyên.

### Thiết lập Airtable

Để sử dụng Airtable, bạn cần:
1. Tạo tài khoản tại [airtable.com](https://airtable.com)
2. Tạo **Personal Access Token** (PAT) trong cài đặt tài khoản
3. Thêm vào file `.env`:

```
AIRTABLE_PAT=pat...
AIRTABLE_BASE_ID=app...
```

4. Chạy script tạo bảng: `python output/tools/airtable.py`

Script `setup()` trong `tools/airtable.py` tự động tạo 2 bảng:
- **Articles** — Lưu tất cả bài viết với các trường: Topic, Status, Word Count, Output File, v.v.
- **Rankings** — Lưu lịch sử vị trí SERP, liên kết với bảng Articles

Nếu chưa cấu hình Airtable, pipeline vẫn hoạt động bình thường — tất cả các hàm đều fallback an toàn (trả về dict rỗng hoặc danh sách rỗng).

In [ ]:
# So sánh SQLite (cũ) vs Airtable (mới)

comparison = [
    ("ID bài viết", "Số nguyên (1, 2, 3)", "Chuỗi (recABC123)"),
    ("Lưu trữ", "File cục bộ (workspace.db)", "Cloud (Airtable)"),
    ("Truy cập", "Chỉ từ máy chạy code", "Từ mọi nơi qua web"),
    ("Giao diện", "Không có (chỉ code)", "Bảng tính trên web"),
    ("Chia sẻ", "Không (file cục bộ)", "Toàn team xem được"),
    ("API", "sqlite3 (Python built-in)", "pyairtable"),
]

print("SQLite (cũ) vs Airtable (mới):")
print(f"{'Tính năng':<20} {'SQLite':<30} {'Airtable':<30}")
print("-" * 80)
for feature, sqlite, airtable in comparison:
    print(f"{feature:<20} {sqlite:<30} {airtable:<30}")

## Database thật của sản phẩm

File `tools/airtable.py` cung cấp các hàm sau:
- `create_article(topic, keywords)` — Tạo bài viết mới (status = 'queued'), trả về record ID (chuỗi)
- `get_article(id)` — Xem chi tiết một bài viết (trả về dict)
- `list_articles()` — Xem tất cả bài viết
- `update_article_status(id, status, **fields)` — Cập nhật trạng thái và nội dung
- `set_published_url(id, url)` — Đặt URL đã xuất bản
- `save_ranking(article_id, keyword, position, ...)` — Lưu kết quả kiểm tra thứ hạng SERP
- `get_rankings(article_id)` — Xem lịch sử thứ hạng
- `setup()` — Tạo bảng Airtable lần đầu (chạy một lần)

Bảng **Articles** có các trường:
- `Topic`, `Status`, `Target Keywords`, `Outline JSON`, `Article Markdown`
- `Output File`, `Word Count`, `Meta Description`, `Batch ID`
- `Published URL` — URL nơi bài viết được đăng (dùng cho theo dõi thứ hạng)
- `Error Message` — Thông báo lỗi nếu pipeline thất bại

Bảng **Rankings** lưu lịch sử vị trí SERP, liên kết với bảng Articles.

Pipeline gọi các hàm này để theo dõi tiến trình:
```
queued -> researching -> outlining -> writing -> enriching -> review -> published (thủ công)
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output"))

from dotenv import load_dotenv
load_dotenv()

from tools.airtable import (
    create_article, get_article, list_articles,
    update_article_status, save_ranking, get_rankings, set_published_url,
)

# Tạo bài viết thử nghiệm
article_id = create_article("Test Article from Notebook", target_keywords=["test", "notebook"])
print(f"Đã tạo bài viết: {article_id}")
print(f"  (Lưu ý: ID là chuỗi, không phải số nguyên)")

# Đọc lại bài viết
article = get_article(article_id)
print(f"\nChi tiết bài viết:")
print(f"  Chủ đề: {article.get('topic', 'N/A')}")
print(f"  Trạng thái: {article.get('status', 'N/A')}")
print(f"  Ngày tạo: {article.get('created_at', 'N/A')}")

# Cập nhật trạng thái
update_article_status(article_id, "review", word_count=1000, article_markdown="# Test\n\nThis is a test.")
article = get_article(article_id)
print(f"\nSau khi cập nhật:")
print(f"  Trạng thái: {article.get('status', 'N/A')}")
print(f"  Số từ: {article.get('word_count', 'N/A')}")

In [ ]:
# Xem tất cả bài viết trong Airtable
articles = list_articles()
print(f"Tổng số bài viết: {len(articles)}\n")
for a in articles:
    print(f"  {a.get('id', 'N/A')}: {a.get('topic', 'N/A')} ({a.get('status', 'N/A')})")

In [ ]:
# Đặt URL đã xuất bản cho bài viết thử nghiệm
set_published_url(article_id, "https://example.com/test-article")
article = get_article(article_id)
print(f"URL đã xuất bản: {article.get('published_url', 'N/A')}")

# Lưu dữ liệu thứ hạng mẫu
save_ranking(article_id, "test keyword", position=15, url="https://example.com/test-article",
             check_date="2026-02-10")
save_ranking(article_id, "test keyword", position=8, url="https://example.com/test-article",
             check_date="2026-02-17")
save_ranking(article_id, "notebook tutorial", position=23, url="https://example.com/test-article",
             check_date="2026-02-17")

# Xem lịch sử thứ hạng
rankings = get_rankings(article_id=article_id)
print(f"\nThứ hạng của bài viết {article_id}:")
for r in rankings:
    pos = r.get('position', 'N/A')
    print(f"  {r.get('check_date', 'N/A')} | {r.get('keyword', 'N/A'):<20} | Vị trí: {pos}")

## Bảng Rankings — Theo dõi thứ hạng SERP

Sau khi xuất bản bài viết, bạn muốn biết: **bài viết có lên được Google không?**

Bảng `rankings` ghi lại vị trí từ khóa theo thời gian. Mỗi hàng là một lần kiểm tra: "Vào ngày này, với từ khóa này, chúng ta đứng vị trí X."

```sql
rankings (id, article_id, keyword, position, url, check_date, search_engine, location)
```

Đây là cách các chuyên gia SEO theo dõi tiến trình — kiểm tra cùng một bộ từ khóa hàng tuần để xem vị trí có cải thiện không.

## Tổng kết

- **Database** lưu trữ dữ liệu lâu dài — dữ liệu vẫn còn khi khởi động lại chương trình
- **Airtable** là database chính — giao diện bảng tính trên cloud, truy cập từ mọi nơi
- **Record ID là chuỗi** (ví dụ: `"recABC123"`), không phải số nguyên — đây là khác biệt quan trọng
- **`tools/airtable.py`** là tầng database của sản phẩm — cung cấp các hàm để tạo, đọc và cập nhật bài viết
- **Theo dõi trạng thái**: Airtable theo dõi từng bài viết qua các bước pipeline (`queued` -> `researching` -> `writing` -> `enriching` -> `review`)
- **SQLite** chỉ còn dùng cho lưu lịch sử chat (`chat_sessions.db`)
- Nhờ có theo dõi này, bạn luôn biết bài viết đang ở bước nào trong quy trình

**Bài tiếp theo:** Chúng ta sẽ khám phá **cách mọi thứ kết nối** — theo dõi hành trình của một yêu cầu từ chat đến pipeline đến database!

## Bài tập

Sử dụng các hàm Airtable đã import ở trên, điền vào chỗ trống để:

1. Tạo một bài viết mới với chủ đề tự chọn
2. Cập nhật trạng thái từ 'queued' sang 'review' với word_count
3. Đọc lại bài viết và kiểm tra trạng thái đã cập nhật
4. Liệt kê tất cả bài viết và đếm số bài ở mỗi trạng thái

Đây chính là các hàm mà pipeline gọi — bạn đang luyện tập nền tảng cơ bản.

In [ ]:
# Bài tập: Điền vào chỗ trống để thao tác với Airtable

# 1. Tạo bài viết mới (điền chủ đề và từ khóa)
new_id = create_article(___, target_keywords=[___, ___])
print(f"Đã tạo bài viết: {new_id}")

# 2. Cập nhật trạng thái sang 'review' với word_count = 1500
update_article_status(new_id, ___, word_count=___)

# 3. Đọc lại bài viết và kiểm tra
updated = get_article(___)
print(f"Trạng thái: {updated.get(___, 'N/A')}")
print(f"Số từ: {updated.get(___, 'N/A')}")

# 4. Liệt kê tất cả và đếm theo trạng thái
all_articles = ___()
status_counts = {}
for a in all_articles:
    s = a.get("status", "unknown")
    status_counts[s] = status_counts.get(s, 0) + 1
print(f"\nTổng: {___(all_articles)} bài viết")
for status, count in status_counts.items():
    print(f"  {status}: {count}")

<details>
<summary>Bấm để xem đáp án</summary>

```python
# 1. Tạo bài viết mới
new_id = create_article("SEO Tips for Beginners", target_keywords=["seo tips", "beginners"])
print(f"Đã tạo bài viết: {new_id}")

# 2. Cập nhật trạng thái sang 'review' với word_count = 1500
update_article_status(new_id, "review", word_count=1500)

# 3. Đọc lại bài viết và kiểm tra
updated = get_article(new_id)
print(f"Trạng thái: {updated.get('status', 'N/A')}")
print(f"Số từ: {updated.get('word_count', 'N/A')}")

# 4. Liệt kê tất cả và đếm theo trạng thái
all_articles = list_articles()
status_counts = {}
for a in all_articles:
    s = a.get("status", "unknown")
    status_counts[s] = status_counts.get(s, 0) + 1
print(f"\nTổng: {len(all_articles)} bài viết")
for status, count in status_counts.items():
    print(f"  {status}: {count}")
```
</details>